# Hybrid parameter learning with Refactored BayesianNetwork

This notebook demonstrates how to:
1. Build a `BayesianNetwork` from `pgmpy/models/Refactored_BayesianNetwork.py` with edges `A -> B`, `B -> C`, and `D -> C`.
2. Use `TabularCPD` for one node and a custom `skpro`-style GAM CPD wrapper for another node.
3. Fit both CPDs together via `HybridEstimator` in `pgmpy.parameter_estimator`.


In [ ]:
import numpy as np
import pandas as pd

from pgmpy.factors.discrete import TabularCPD
from pgmpy.models.Refactored_BayesianNetwork import BayesianNetwork
from pgmpy.parameter_estimator import HybridEstimator


In [ ]:
# 1) Create the structure: A -> B, B -> C, D -> C
model = BayesianNetwork(ebunch=[("A", "B"), ("B", "C"), ("D", "C")])
model.nodes(), model.edges()


In [ ]:
# 2) Define CPDs/models
# Discrete CPD for B conditioned on A.
cpd_b = TabularCPD(
    variable="B",
    variable_card=2,
    values=[[0.8, 0.3], [0.2, 0.7]],
    evidence=["A"],
    evidence_card=[2],
)

class SkproGAMCPD:
    """CPD-like wrapper around an skpro GAM model for node C.

    This wrapper exposes `variable`, `get_evidence`, and `fit(data)` so that
    it can be consumed by `HybridEstimator`.
    """

    def __init__(self, variable, evidence):
        self.variable = variable
        self._evidence = list(evidence)
        self.model = None

    @property
    def variables(self):
        return [self.variable, *self._evidence]

    @property
    def cardinality(self):
        # Not used for continuous model in this demo.
        return []

    def get_evidence(self):
        return self._evidence

    def fit(self, data):
        X = data[self._evidence]
        y = data[self.variable]

        # Example skpro GAM regressor import.
        # If your installed skpro version exposes GAM at a different path,
        # adapt this import accordingly.
        from skpro.regression.gam import GAMRegressor

        self.model = GAMRegressor()
        self.model.fit(X, y)
        return self

cpd_c = SkproGAMCPD(variable="C", evidence=["B", "D"])


In [ ]:
# A and D are root nodes; for this hybrid estimation demo,
# we fit only CPD-like objects we provide in model.cpds.
model.add_cpds(cpd_b, cpd_c)

# Optional check for B and C parent consistency
model.get_cpds("B").get_evidence(), model.get_cpds("C").get_evidence()


In [ ]:
# 3) Generate synthetic training data
rng = np.random.default_rng(42)
n = 500

A = rng.integers(0, 2, size=n)
D = rng.normal(0.0, 1.0, size=n)

# Sample B from the TabularCPD probabilities
p_B1_given_A = np.where(A == 0, 0.2, 0.7)
B = (rng.random(n) < p_B1_given_A).astype(int)

# Continuous C depends nonlinearly on B and D
C = 2.0 * B + np.sin(D) + 0.3 * (D ** 2) + rng.normal(0.0, 0.2, size=n)

data = pd.DataFrame({"A": A, "B": B, "C": C, "D": D})
data.head()


In [ ]:
# 4) Fit both CPDs/models with HybridEstimator
estimator = HybridEstimator()
estimator.fit(model=model, data=data)

len(estimator.parameters_), [type(p).__name__ for p in estimator.parameters_]


In [ ]:
# Inspect fitted outputs
fitted_b = estimator.parameters_[0]
fitted_c = estimator.parameters_[1]

print(fitted_b)
print("\nFitted skpro GAM model:", fitted_c.model)
